
# V42 LIVE — Dedicated Sarcasm Specialist

**Protected public-LB champion:** V38 = **0.608**  
**Target:** **0.620**

V42 does **not** touch the three components already proved useful on Kaggle:

- `is_toxic` — frozen from V38
- `implicit_toxicity` — frozen from V38
- `threatening` — frozen from V38

It also keeps the V14/V38 versions of:

- `contains_profanity`
- `identity_attack`
- `sexual_content`

V42 trains a **new sarcasm-only system** from the exact Kaggle-LIVE 3297/500/1000 split.

## Architecture

Two complementary experts:

1. **Twitter-XLM-R** (`cardiffnlp/twitter-xlm-roberta-base`)
   - multilingual
   - pretrained on social-media language
   - useful for Banglish/code-mixed text

2. **BanglaBERT + BiLSTM**
   - `csebuetnlp/banglabert`
   - contextual Bengali encoder
   - BiLSTM sequence head for sarcasm cues

Training uses:

- exact-duplicate grouped 5-fold CV
- external 500-row validation kept fully untouched during model training
- layer-wise learning-rate decay (LLRD)
- mild positive oversampling
- BCE + pairwise ranking loss
- FGM adversarial embedding training
- epoch-2/3 snapshot averaging
- script-aware ensemble learned **only from train OOF**
- checkpoint/resume support

The notebook creates several candidate CSVs and a result ZIP.  
**Do not submit a candidate before reviewing `v42_results.zip`.**


In [ ]:

# 1. Upload V42 data bundle
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v42_LIVE_sarcasm_specialist_data.zip" in uploaded

DATA_DIR = pathlib.Path("/content/v42_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v42_LIVE_sarcasm_specialist_data.zip") as z:
    z.extractall(DATA_DIR)

for p in sorted(DATA_DIR.iterdir()):
    print(p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install dependencies and preflight model access
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn>=1.4",
    "safetensors",
    "tqdm",
])

from transformers import AutoConfig, AutoTokenizer

MODEL_TWITTER = "cardiffnlp/twitter-xlm-roberta-base"
MODEL_BANGLA = "csebuetnlp/banglabert"

for m in [MODEL_TWITTER, MODEL_BANGLA]:
    print("Checking:", m)
    _ = AutoConfig.from_pretrained(m)
    _ = AutoTokenizer.from_pretrained(m)

print("✅ Both V42 checkpoints are accessible.")


In [ ]:

# 3. Imports, configuration, optional Drive checkpointing
import gc, re, math, json, random, unicodedata, hashlib, os
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, average_precision_score
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

SEED = 2026
N_FOLDS = 5
EPOCHS = 3
SNAPSHOT_EPOCHS = [2, 3]
MAX_LENGTH = 128

TRAIN_BATCH = 6
EVAL_BATCH = 32
ACCUM = 3

BASE_LR_TWITTER = 1.5e-5
BASE_LR_BANGLA = 1.3e-5
HEAD_MULT = 3.0
LAYER_DECAY = 0.90
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

RANK_WEIGHT = 0.18
FGM_WEIGHT = 0.35
FGM_EPS = 0.55

MODEL_TWITTER = "cardiffnlp/twitter-xlm-roberta-base"
MODEL_BANGLA = "csebuetnlp/banglabert"

TARGET = "sarcasm"
LABELS = [
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]

USE_DRIVE_CHECKPOINTS = True

if USE_DRIVE_CHECKPOINTS:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/V42_Bengali_NLP_Checkpoints")
else:
    RUN_DIR = Path("/content/v42_output")

RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/content/v42_export")
EXPORT_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(SEED)
assert torch.cuda.is_available(), "Enable a T4 GPU in Colab."

print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint directory:", RUN_DIR)


In [ ]:

# 4. Load and verify the exact Kaggle-LIVE split
train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2v = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2t = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v38_champion.csv", dtype={"ID":str})

assert len(train) == 3297
assert len(valid) == 500
assert len(test) == 1000
assert test["ID"].tolist() == champ["ID"].tolist()
assert valid["ID"].tolist() == v2v["ID"].tolist()
assert test["ID"].tolist() == v2t["ID"].tolist()

assert test["ID"].head(4).tolist() == [
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

def clean_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.replace("\u200c"," ").replace("\u200d"," ")
    x = re.sub(r"https?://\S+|www\.\S+", " URL ", x)
    x = re.sub(r"@\w+", " USER ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

BN_RE = re.compile(r"[\u0980-\u09FF]")
LAT_RE = re.compile(r"[A-Za-z]")

def script_type(x):
    s = str(x)
    bn = bool(BN_RE.search(s))
    lat = bool(LAT_RE.search(s))
    if bn and lat:
        return "mixed"
    if bn:
        return "native"
    return "roman"

for df in [train, valid, test]:
    df["clean_text"] = df["text"].map(clean_text)
    df["group_text"] = df["clean_text"].str.lower()
    df["script"] = df["clean_text"].map(script_type)

print("✅ Exact accepted LIVE IDs verified.")
print()
print("Sarcasm prevalence:")
print(" train:", int(train[TARGET].sum()), "/", len(train), "=", round(train[TARGET].mean(),4))
print(" valid:", int(valid[TARGET].sum()), "/", len(valid), "=", round(valid[TARGET].mean(),4))
print(" V38 test positives:", int(champ[TARGET].sum()), "/", len(test))

display(pd.DataFrame({
    "train": train["script"].value_counts(),
    "valid": valid["script"].value_counts(),
    "test": test["script"].value_counts(),
}).fillna(0).astype(int))


In [ ]:

# 5. Grouped 5-fold split on TRAIN only
#
# The supplied 500-row validation set is NOT used to train or choose
# any neural-model parameters or ensemble weights.

sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

train["fold"] = -1

for fold, (_, va_idx) in enumerate(
    sgkf.split(
        train,
        y=train[TARGET],
        groups=train["group_text"]
    )
):
    train.loc[va_idx, "fold"] = fold

assert (train["fold"] >= 0).all()
assert train.groupby("group_text")["fold"].nunique().max() == 1

audit = train.groupby("fold").agg(
    rows=("ID","size"),
    sarcasm_pos=(TARGET,"sum"),
)
audit["rate"] = audit["sarcasm_pos"] / audit["rows"]

display(audit)


In [ ]:

# 6. Dataset, collator, models and training helpers

class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, labels=True):
        self.text = df["clean_text"].astype(str).tolist()
        self.y = df[TARGET].to_numpy(np.float32) if labels else None
        self.tok = tokenizer

    def __len__(self):
        return len(self.text)

    def __getitem__(self, i):
        out = self.tok(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            out["labels"] = self.y[i]
        return out


class PadCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, items):
        labels = None
        if "labels" in items[0]:
            labels = torch.tensor(
                [float(x.pop("labels")) for x in items],
                dtype=torch.float32
            )

        batch = self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )

        if labels is not None:
            batch["labels"] = labels

        return batch


class BanglaBertBiLSTM(nn.Module):
    def __init__(self, model_name, lstm_hidden=128, dropout=0.20):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size

        self.lstm = nn.LSTM(
            input_size=h,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_hidden*2, 1)

        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

    def forward(self, input_ids, attention_mask, **kwargs):
        enc_inputs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        if "token_type_ids" in kwargs:
            enc_inputs["token_type_ids"] = kwargs["token_type_ids"]

        h = self.encoder(**enc_inputs).last_hidden_state
        h, _ = self.lstm(h)

        mask = attention_mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1).clamp_min(1.0)

        logits = self.classifier(
            self.dropout(pooled)
        ).squeeze(-1)

        return logits


def make_model(kind):
    if kind == "twitter":
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_TWITTER,
            num_labels=1,
        )
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        return model

    if kind == "bangla_bilstm":
        return BanglaBertBiLSTM(MODEL_BANGLA)

    raise ValueError(kind)


def forward_logits(model, batch):
    inputs = {
        k:v for k,v in batch.items()
        if k != "labels"
    }

    out = model(**inputs)

    if torch.is_tensor(out):
        return out

    return out.logits.squeeze(-1)


def pairwise_rank_loss(logits, y):
    pos = logits[y > 0.5]
    neg = logits[y <= 0.5]

    if len(pos) == 0 or len(neg) == 0:
        return logits.new_tensor(0.0)

    # Limit pair matrix for stability and speed.
    if len(pos) > 16:
        pos = pos[:16]
    if len(neg) > 32:
        neg = neg[:32]

    diff = pos[:,None] - neg[None,:]
    return torch.nn.functional.softplus(-diff).mean()


class FGM:
    def __init__(self, model, epsilon=0.55):
        self.model = model
        self.epsilon = epsilon
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if (
                param.requires_grad
                and param.grad is not None
                and (
                    "word_embeddings" in name
                    or "embeddings.word" in name
                )
            ):
                norm = torch.norm(param.grad)
                if torch.isfinite(norm) and norm > 0:
                    self.backup[name] = param.data.clone()
                    param.data.add_(
                        self.epsilon * param.grad / norm
                    )

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


def build_llrd_optimizer(model, base_lr):
    named = list(model.named_parameters())

    layer_ids = []
    rx = re.compile(r"encoder\.layer\.(\d+)")

    for name, _ in named:
        m = rx.search(name)
        if m:
            layer_ids.append(int(m.group(1)))

    max_layer = max(layer_ids) if layer_ids else 11

    groups = {}

    for name, p in named:
        if not p.requires_grad:
            continue

        is_head = (
            "classifier" in name
            or "lstm" in name
            or name.startswith("score")
        )

        m = rx.search(name)

        if is_head:
            lr = base_lr * HEAD_MULT
        elif m:
            layer = int(m.group(1))
            lr = base_lr * (LAYER_DECAY ** (max_layer-layer))
        else:
            lr = base_lr * (LAYER_DECAY ** (max_layer+1))

        no_decay = (
            name.endswith("bias")
            or "LayerNorm.weight" in name
            or "layer_norm.weight" in name
        )

        wd = 0.0 if no_decay else WEIGHT_DECAY
        key = (round(lr, 12), wd)

        groups.setdefault(key, []).append(p)

    params = [
        {"params": ps, "lr": lr, "weight_decay": wd}
        for (lr, wd), ps in groups.items()
    ]

    return torch.optim.AdamW(params)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []

    for batch in loader:
        batch.pop("labels", None)

        batch = {
            k:v.to("cuda", non_blocking=True)
            for k,v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = forward_logits(model, batch)

        out.append(
            torch.sigmoid(
                logits.float()
            ).cpu().numpy()
        )

    return np.concatenate(out)


In [ ]:

# 7. Train one expert with 5-fold OOF + untouched validation/test prediction

EXPERTS = {
    "twitter": {
        "model_name": MODEL_TWITTER,
        "base_lr": BASE_LR_TWITTER,
    },
    "bangla_bilstm": {
        "model_name": MODEL_BANGLA,
        "base_lr": BASE_LR_BANGLA,
    },
}

fold_seeds = [2026, 17, 3407, 1337, 42]


def train_expert(kind):
    cfg = EXPERTS[kind]

    tokenizer = AutoTokenizer.from_pretrained(
        cfg["model_name"]
    )

    collator = PadCollator(tokenizer)

    oof = np.zeros(len(train), np.float32)
    valid_parts = []
    test_parts = []
    history = []

    for fold in range(N_FOLDS):
        tr = train[train["fold"] != fold].copy()
        va = train[train["fold"] == fold].copy()

        cache = RUN_DIR / f"{kind}_fold_{fold}.npz"
        hist_path = RUN_DIR / f"{kind}_fold_{fold}_history.csv"

        if cache.exists():
            try:
                with np.load(cache) as z:
                    idx = z["idx"]
                    pva = z["oof"]
                    pev = z["external_valid"]
                    pte = z["test"]

                if np.array_equal(idx, va.index.to_numpy()):
                    print("✅ reuse", kind, "fold", fold)
                    oof[idx] = pva
                    valid_parts.append(pev)
                    test_parts.append(pte)

                    if hist_path.exists():
                        history.extend(
                            pd.read_csv(hist_path).to_dict("records")
                        )

                    continue
            except Exception as e:
                print("Cache ignored:", e)

        seed_all(fold_seeds[fold])

        # Mild oversampling. Do not force a 50/50 training distribution.
        sample_weights = np.where(
            tr[TARGET].to_numpy(int) == 1,
            2.0,
            1.0
        )

        sampler = WeightedRandomSampler(
            weights=torch.tensor(
                sample_weights,
                dtype=torch.double
            ),
            num_samples=len(tr),
            replacement=True,
        )

        tr_loader = DataLoader(
            SarcasmDataset(tr, tokenizer, True),
            batch_size=TRAIN_BATCH,
            sampler=sampler,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )

        va_loader = DataLoader(
            SarcasmDataset(va, tokenizer, True),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )

        ext_loader = DataLoader(
            SarcasmDataset(valid, tokenizer, True),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )

        test_loader = DataLoader(
            SarcasmDataset(test, tokenizer, False),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )

        gc.collect()
        torch.cuda.empty_cache()

        model = make_model(kind).to("cuda")

        optimizer = build_llrd_optimizer(
            model,
            cfg["base_lr"]
        )

        steps = math.ceil(len(tr_loader)/ACCUM) * EPOCHS

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            int(steps*WARMUP_RATIO),
            steps,
        )

        scaler = torch.amp.GradScaler("cuda")
        fgm = FGM(model, FGM_EPS)

        fold_oof_snaps = []
        fold_ext_snaps = []
        fold_test_snaps = []
        fold_hist = []

        for epoch in range(1, EPOCHS+1):
            model.train()
            optimizer.zero_grad(set_to_none=True)

            running = 0.0
            seen = 0

            for step, batch in enumerate(
                tqdm(
                    tr_loader,
                    desc=f"{kind} f{fold} e{epoch}/{EPOCHS}"
                )
            ):
                y = batch.pop("labels").to(
                    "cuda",
                    non_blocking=True
                )

                inputs = {
                    k:v.to("cuda", non_blocking=True)
                    for k,v in batch.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    logits = forward_logits(model, inputs)

                bce = torch.nn.functional.binary_cross_entropy_with_logits(
                    logits.float(),
                    y
                )

                rank = pairwise_rank_loss(
                    logits.float(),
                    y
                )

                loss = bce + RANK_WEIGHT*rank

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        f"non-finite loss: {kind} fold {fold}"
                    )

                scaler.scale(
                    loss/ACCUM
                ).backward()

                # FGM only after epoch 1. This reduces early instability.
                if epoch >= 2:
                    fgm.attack()

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):
                        adv_logits = forward_logits(
                            model,
                            inputs
                        )

                    adv_bce = torch.nn.functional.binary_cross_entropy_with_logits(
                        adv_logits.float(),
                        y
                    )

                    adv_rank = pairwise_rank_loss(
                        adv_logits.float(),
                        y
                    )

                    adv_loss = adv_bce + RANK_WEIGHT*adv_rank

                    scaler.scale(
                        FGM_WEIGHT*adv_loss/ACCUM
                    ).backward()

                    fgm.restore()

                running += float(loss.item()) * len(y)
                seen += len(y)

                if (
                    (step+1) % ACCUM == 0
                    or step+1 == len(tr_loader)
                ):
                    scaler.unscale_(optimizer)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        1.0
                    )

                    old_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    if scaler.get_scale() >= old_scale:
                        scheduler.step()

                    optimizer.zero_grad(set_to_none=True)

            rec = {
                "expert": kind,
                "fold": fold,
                "epoch": epoch,
                "loss": running/max(seen,1),
            }

            fold_hist.append(rec)
            history.append(rec)

            print(rec)

            pd.DataFrame(fold_hist).to_csv(
                hist_path,
                index=False
            )

            if epoch in SNAPSHOT_EPOCHS:
                fold_oof_snaps.append(
                    predict(model, va_loader)
                )

                fold_ext_snaps.append(
                    predict(model, ext_loader)
                )

                fold_test_snaps.append(
                    predict(model, test_loader)
                )

        pva = np.mean(fold_oof_snaps, axis=0)
        pev = np.mean(fold_ext_snaps, axis=0)
        pte = np.mean(fold_test_snaps, axis=0)

        idx = va.index.to_numpy()
        oof[idx] = pva
        valid_parts.append(pev)
        test_parts.append(pte)

        np.savez_compressed(
            cache,
            idx=idx,
            oof=pva,
            external_valid=pev,
            test=pte,
        )

        del model, optimizer, scheduler, scaler
        del tr_loader, va_loader, ext_loader, test_loader
        gc.collect()
        torch.cuda.empty_cache()

    external_valid = np.mean(valid_parts, axis=0)
    test_prob = np.mean(test_parts, axis=0)

    np.savez_compressed(
        RUN_DIR/f"{kind}_complete.npz",
        oof=oof,
        external_valid=external_valid,
        test=test_prob,
    )

    pd.DataFrame(history).to_csv(
        RUN_DIR/f"{kind}_training_history.csv",
        index=False
    )

    return oof, external_valid, test_prob


In [ ]:

# 8. Run both sarcasm experts

twitter_oof, twitter_valid, twitter_test = train_expert(
    "twitter"
)

bangla_oof, bangla_valid, bangla_test = train_expert(
    "bangla_bilstm"
)

print("✅ Both experts trained.")


In [ ]:

# 9. Train-OOF-only script-aware ensemble recipe
#
# No external-validation labels are used in this cell.

def logit(p):
    p = np.clip(np.asarray(p,float), 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

def f1_pos(y,p):
    y = np.asarray(y,int)
    p = np.asarray(p,int)

    tp = ((y==1)&(p==1)).sum()
    fp = ((y==0)&(p==1)).sum()
    fn = ((y==1)&(p==0)).sum()

    d = 2*tp + fp + fn
    return float(2*tp/d) if d else 0.0

def topk_local(prob, k):
    out = np.zeros(len(prob),int)
    k = int(max(0,min(k,len(prob))))

    if k:
        out[np.argsort(-prob)[:k]] = 1

    return out

def script_blend(
    twitter_prob,
    bangla_prob,
    script_values,
    recipe
):
    out = np.zeros(len(twitter_prob),float)

    for script in ["native","mixed","roman"]:
        m = np.asarray(script_values) == script
        w_bangla = recipe[script]["w_bangla"]

        out[m] = sigmoid(
            w_bangla*logit(bangla_prob[m])
            +(1-w_bangla)*logit(twitter_prob[m])
        )

    return out


recipe = {}
y_train = train[TARGET].to_numpy(int)
folds = train["fold"].to_numpy(int)

for script in ["native","mixed","roman"]:
    mask = train["script"].to_numpy() == script

    if mask.sum() == 0:
        continue

    # Pick blend weight by TRAIN OOF average precision.
    candidates = []

    for w_bangla in [0.0,0.25,0.50,0.75,1.0]:
        score = sigmoid(
            w_bangla*logit(bangla_oof)
            +(1-w_bangla)*logit(twitter_oof)
        )

        ap = average_precision_score(
            y_train[mask],
            score[mask]
        )

        candidates.append(
            (ap,w_bangla,score)
        )

    candidates.sort(reverse=True, key=lambda x:x[0])

    best_ap, w_bangla, blended = candidates[0]

    # Estimate the positive prediction rate independently in each
    # OOF fold; use the median to reduce one-fold threshold noise.
    fold_rates = []
    fold_f1 = []

    for f in range(N_FOLDS):
        fm = mask & (folds == f)

        if fm.sum() == 0:
            continue

        yy = y_train[fm]
        pp = blended[fm]

        best = (-1,None)

        # Coarse rate grid prevents threshold overfitting.
        for rate in np.arange(0.08,0.91,0.04):
            k = int(round(rate*fm.sum()))
            pred = topk_local(pp,k)
            sc = f1_pos(yy,pred)

            if sc > best[0]:
                best = (sc,float(rate))

        fold_f1.append(best[0])
        fold_rates.append(best[1])

    median_rate = float(np.median(fold_rates))
    empirical = float(y_train[mask].mean())

    # 20% shrinkage toward empirical prevalence.
    final_rate = float(
        0.80*median_rate
        +0.20*empirical
    )

    recipe[script] = {
        "w_bangla": float(w_bangla),
        "oof_ap": float(best_ap),
        "fold_optimal_rates": fold_rates,
        "median_rate": median_rate,
        "empirical_rate": empirical,
        "final_rate": final_rate,
        "mean_fold_f1": float(np.mean(fold_f1)),
        "std_fold_f1": float(np.std(fold_f1)),
    }

print(json.dumps(recipe, indent=2))


In [ ]:

# 10. Build validation/test candidate predictions

new_valid_score = script_blend(
    twitter_valid,
    bangla_valid,
    valid["script"].to_numpy(),
    recipe
)

new_test_score = script_blend(
    twitter_test,
    bangla_test,
    test["script"].to_numpy(),
    recipe
)

# Candidate A: rate learned entirely from TRAIN OOF.
pred_valid_cv = np.zeros(len(valid),int)
pred_test_cv = np.zeros(len(test),int)

for script in ["native","mixed","roman"]:
    mv = valid["script"].to_numpy() == script
    mt = test["script"].to_numpy() == script

    rate = recipe[script]["final_rate"]

    pred_valid_cv[mv] = topk_local(
        new_valid_score[mv],
        int(round(rate*mv.sum()))
    )

    pred_test_cv[mt] = topk_local(
        new_test_score[mt],
        int(round(rate*mt.sum()))
    )


# Candidate B: conservative rerank preserving V38 positive counts.
#
# Validation count is the V2/V14 validation prediction count because
# V38 never changed sarcasm.
BASE_THRESHOLD = 0.175

base_valid_pred = (
    v2v["sarcasm_prob"].to_numpy(float)
    >= BASE_THRESHOLD
).astype(int)

base_test_pred = champ["sarcasm"].to_numpy(int)

pred_valid_fixed = topk_local(
    new_valid_score,
    int(base_valid_pred.sum())
)

pred_test_fixed = topk_local(
    new_test_score,
    int(base_test_pred.sum())
)


# Candidate C: 50/50 new specialist score + old V2 rank,
# fixed V38 positive counts.
blend_valid_score = sigmoid(
    .50*logit(new_valid_score)
    +.50*logit(v2v["sarcasm_prob"].to_numpy(float))
)

blend_test_score = sigmoid(
    .50*logit(new_test_score)
    +.50*logit(v2t["sarcasm_prob"].to_numpy(float))
)

pred_valid_blend = topk_local(
    blend_valid_score,
    int(base_valid_pred.sum())
)

pred_test_blend = topk_local(
    blend_test_score,
    int(base_test_pred.sum())
)


In [ ]:

# 11. Untouched 500-row validation gate
#
# This is the FIRST place where validation labels are used.

y_valid = valid[TARGET].to_numpy(int)

baseline_f1 = f1_pos(
    y_valid,
    base_valid_pred
)

baseline_ap = average_precision_score(
    y_valid,
    v2v["sarcasm_prob"].to_numpy(float)
)

candidates = {
    "cv_rate": (
        pred_valid_cv,
        pred_test_cv,
        new_valid_score,
    ),
    "fixed_v38_count": (
        pred_valid_fixed,
        pred_test_fixed,
        new_valid_score,
    ),
    "blend50_fixed_count": (
        pred_valid_blend,
        pred_test_blend,
        blend_valid_score,
    ),
}

rows = []

for name,(pv,pt,scorev) in candidates.items():
    rows.append({
        "candidate": name,
        "validation_f1": f1_pos(y_valid,pv),
        "validation_gain": f1_pos(y_valid,pv)-baseline_f1,
        "validation_ap": average_precision_score(y_valid,scorev),
        "validation_positive_count": int(pv.sum()),
        "test_positive_count": int(pt.sum()),
        "test_changes_vs_v38": int(
            (pt != base_test_pred).sum()
        ),
    })

report = pd.DataFrame(rows).sort_values(
    ["validation_f1","validation_ap"],
    ascending=False
).reset_index(drop=True)

print("V38/V14 sarcasm validation F1:", baseline_f1)
print("V38/V14 sarcasm validation AP:", baseline_ap)
display(report.round(6))

best = report.iloc[0].to_dict()

# Strong gate because local improvements have historically failed to transfer.
PASS = bool(
    best["validation_gain"] >= 0.035
    and best["validation_ap"] >= baseline_ap + 0.025
)

summary = {
    "protected_public_lb": 0.608,
    "target_public_lb": 0.620,
    "baseline_validation_f1": float(baseline_f1),
    "baseline_validation_ap": float(baseline_ap),
    "best_candidate": best,
    "PASS": PASS,
    "recipe": recipe,
}

print(json.dumps(summary, indent=2))


In [ ]:

# 12. Create V42 candidate submissions
#
# All six non-sarcasm columns remain EXACTLY V38.

candidate_test_map = {
    "cv_rate": pred_test_cv,
    "fixed_v38_count": pred_test_fixed,
    "blend50_fixed_count": pred_test_blend,
}

submission_paths = {}

for name, sarcasm_pred in candidate_test_map.items():
    sub = champ.copy()
    sub["sarcasm"] = sarcasm_pred.astype(int)

    assert sub["ID"].tolist() == test["ID"].tolist()
    assert sub["ID"].head(4).tolist() == [
        "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
    ]

    for label in LABELS:
        if label != "sarcasm":
            assert np.array_equal(
                sub[label].to_numpy(int),
                champ[label].to_numpy(int)
            )

    assert sub[LABELS].isin([0,1]).all().all()

    path = EXPORT_DIR/f"submission_v42_{name}.csv"
    sub.to_csv(path,index=False)
    submission_paths[name] = str(path)

# Recommended file only if strong gate passes.
recommended_name = report.iloc[0]["candidate"]

if PASS:
    recommended = pd.read_csv(
        submission_paths[recommended_name],
        dtype={"ID":str}
    )

    recommended_path = (
        EXPORT_DIR/"submission_v42_RECOMMENDED.csv"
    )
    recommended.to_csv(recommended_path,index=False)

    print("✅ V42 PASSED.")
    print("Recommended:", recommended_name)
    print("File:", recommended_path)
else:
    print("❌ V42 DID NOT PASS THE STRICT GATE.")
    print("Do NOT submit yet. Send v42_results.zip for analysis.")


In [ ]:

# 13. Save probabilities, reports and package results

np.savez_compressed(
    EXPORT_DIR/"v42_sarcasm_probabilities.npz",
    twitter_oof=twitter_oof,
    twitter_valid=twitter_valid,
    twitter_test=twitter_test,
    bangla_oof=bangla_oof,
    bangla_valid=bangla_valid,
    bangla_test=bangla_test,
    new_valid_score=new_valid_score,
    new_test_score=new_test_score,
)

report.to_csv(
    EXPORT_DIR/"v42_validation_report.csv",
    index=False
)

with open(
    EXPORT_DIR/"v42_summary.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

with open(
    EXPORT_DIR/"v42_recipe.json",
    "w"
) as f:
    json.dump(
        recipe,
        f,
        indent=2
    )

zip_path = Path("/content/v42_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
